# Lab: Greedy Best-First Search (GBFS)
## Case Study: Cybersecurity – Lateral Movement in Enterprise Network

### Scenario

A security operations team is investigating how an attacker moved through an enterprise network.

The attacker initially compromised an internet-facing machine and attempted to reach the critical database server (Crown Jewel Asset).

We model the enterprise network as a graph:

- Nodes → Machines (servers, workstations, gateways)
- Edges → Network connectivity (possible pivot paths)
- Start → Initial compromised host
- Goal → Critical Database Server (DB-Core)

We simulate attacker behavior using Greedy Best-First Search (GBFS).

Evaluation Function:
    f(n) = h(n)

GBFS selects the node with the smallest heuristic value.

It does NOT consider pthe actual path cost g(n) or the detection risk.

## Learning Objectives

1. Model enterprise lateral movement as a graph search problem.
2. Implement Greedy Best-First Search using a priority queue.
3. Analyze how heuristic bias affects attacker simulation.
4. Observe how GBFS can choose risky or suboptimal attack paths.
5. Compare later with cost-aware search.

## Heuristic Interpretation

The heuristic value h(n) represents the estimated privilege proximity to DB-Core

Lower value → Closer to critical asset  
Higher value → Further from critical asset

## Step 1: Import Required Libraries

We use heapq to implement a priority queue
for selecting the node with the lowest heuristic value.

In [24]:
# Making necessary imports
import heapq

## Step 2: Load Network Graph from File

The network file contains pairs of connected machines:

Format:

    NodeA NodeB
  
The graph is treated as undirected because
lateral movement is possible in both directions.

In [25]:
def load_network(filename):
    graph = {}

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue

            a, b = line.strip().split()

            #Populate the graph
            if a not in graph:
                graph[a] = []
            graph[a].append(b)

    return graph

## Step 3: Load Heuristic Values

The heuristic file contains:

    Node HeuristicValue

These values estimate how close a node is to DB-Core.

In [26]:
def load_heuristic(filename):
    heuristic = {}

    # TODO: Open file
    # TODO: Read node and heuristic value
    # TODO: Store in dictionary

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue

            a, b = line.strip().split()

            heuristic[a] = b

    return heuristic

## Step 4: Implement Greedy Best-First Search

Algorithm:

1. Insert start node into priority queue with priority h(start)
2. While frontier is not empty:
   - Always expand the node with the smallest heuristic h(n)
   - If node is goal, stop
   - Mark node as visited
   - Add unvisited neighbors to priority queue
   - Track parent pointers for path reconstruction.



In [27]:
def greedy_best_first_search(graph, heuristic, start, goal):
    """
    Implements GBFS algorithm.

    Returns:
        path (list)
        total_cost (int)
        explored_order (list)
    """

    # Priority queue: (heuristic_value, node)
    frontier = []

    # TODO: Push start node into priority queue
    heapq.heappush(frontier, (heuristic[start], start))

    visited = set()

    parent = {}  # To reconstruct path

    parent[start] = None 
    
    explored_order = []

    while frontier:

        # TODO: Pop node with lowest heuristic value
        _, current = heapq.heappop(frontier)

        # TODO: Add node to explored_order
        explored_order.append(current)

        # TODO: Check if goal reached
        if current == goal:
            break

        # TODO: Mark node as visited
        visited.add(current)

        # TODO: Expand neighbors
        # For each neighbor:
        #   If not visited:
        #       Set parent
        #       Push into priority queue with heuristic value

        for neigh in graph[current]:
            if neigh not in visited:
                parent[neigh] = current
                heapq.heappush(frontier, (heuristic[neigh], neigh))

    # Goal found
    if explored_order[-1] == goal:
        path = reconstruct_path(parent, start, goal)
        cost = calculate_cost(graph, path)

        return path, cost, explored_order
        
    # TODO: If goal not found, return None
    return None, None, explored_order

## Step 5: Path Reconstruction

We backtrack from the goal to the start using the parent dictionary.

In [28]:
def reconstruct_path(parent, start, goal):
    """
    Reconstructs path from start to goal.
    """

    path = []

    # TODO: Backtrack from goal to start using parent dictionary
    
    cur_node = goal

    while parent[cur_node]:
        path.append(cur_node)
        cur_node = parent[cur_node]

    path.append(start)

    return path[::-1]

##Step 5: Calculate Path Cost
Although GBFS ignores cost during search, we compute the actual cost afterward for evaluation.

In [29]:
def calculate_cost(graph, path):
    """
    Calculates total cost of path.
    """

    total_cost = 0

    # TODO: Loop through path and sum edge costs
    # Straight unweighted path

    total_cost = len(path) - 1

    return total_cost


##Step 6: Execute the Algorithm - Do NOT alter the following code

In [30]:
def save_output(filename, explored, path, cost):

    with open(filename, 'w') as file:
        file.write("Greedy Best-First Search Results\n")
        file.write("--------------------------------\n\n")

        file.write("Exploration Order:\n")
        file.write(str(explored) + "\n\n")

        if path is None:
            file.write("Simulated Attack Path: None\n")
            file.write("Total Cost: None\n")
        else:
            file.write("Simulated Attack Path:\n")
            file.write(" -> ".join(path) + "\n\n")
            file.write("Total Cost:\n")
            file.write(str(cost) + "\n")

    print(f"Results saved to {filename}")

if __name__ == "__main__":

    graph = load_network("network.txt")
    heuristic = load_heuristic("heuristic.txt")
    # graph = load_network("network_test.txt")
    # heuristic = load_heuristic("heuristic_test.txt")

    start = "Web-Gateway"
    goal = "DB-Core"

    path, cost, explored = greedy_best_first_search(graph, heuristic, start, goal)

    print("\nExploration Order:", explored)
    print("Simulated Attack Path:", path)
    print("Total Cost:", cost)

    save_output("output.txt", explored, path, cost)


Exploration Order: ['Web-Gateway', 'App-Server', 'Mail-Server', 'Dev-Workstation', 'Sandbox', 'Research-PC', 'Admin-Console', 'Logging-Server', 'Jump-Host', 'Domain-Controller', 'DB-Core']
Simulated Attack Path: ['Web-Gateway', 'App-Server', 'Admin-Console', 'Logging-Server', 'Jump-Host', 'Domain-Controller', 'DB-Core']
Total Cost: 6
Results saved to output.txt


##Conclusion
On a piece of paper, draw the graph provided in the network.txt file. What is the optimal path? Did your algorithm find the Optimal Path? Why or why not?

In [31]:
# Answer here:
# for network.txt and heuristic.txt
# There are 4 paths and the path choosen here:
# ['Web-Gateway', 'App-Server', 'Admin-Console', 'Logging-Server', 'Jump-Host', 'Domain-Controller', 'DB-Core']
# have cost 6

# while all others path have cost 5 so the path choosen here is NOT OPTIMAL

# WHy not optimal choosen

# G BFS chooses path based on shorted heuristic of neighbours of current node.
# SInce the start (Web-Gateway) have neighbours App-Server with 11 heuristic and Dev-Workstation with 15 so it chooses App-Server

# Similarly, for App-Server node it chooses Admin-Console (5) instead of Mailing-Server (10) and goes to un optimal path